# 05 — Inference Demo

This notebook connects Model 1 and Model 2 into a single prediction flow.

1. Estimate machine-failure probability.
2. Apply the selected Model 1 threshold.
3. If failure is predicted, identify one or more failure types.
4. Convert the result into a project-level risk/action rule.

The risk rules below are **project rules**, not industry standards.

In [1]:
import os
import joblib
import pandas as pd
import numpy as np

MODEL_DIR = "../models"

model1 = joblib.load(os.path.join(MODEL_DIR, "model_failure_rf.joblib"))
prep1 = joblib.load(os.path.join(MODEL_DIR, "model_failure_preprocessor.joblib"))
meta1 = joblib.load(os.path.join(MODEL_DIR, "model_failure_metadata.joblib"))

model2 = joblib.load(os.path.join(MODEL_DIR, "model_failure_types_rf.joblib"))
prep2 = joblib.load(os.path.join(MODEL_DIR, "model_failure_types_preprocessor.joblib"))
meta2 = joblib.load(os.path.join(MODEL_DIR, "model_failure_types_metadata.joblib"))

print("Models loaded successfully.")
print("Model 1 threshold:", meta1["threshold"])
print("Model 2 labels:", meta2["labels"])


Models loaded successfully.
Model 1 threshold: 0.3500000000000001
Model 2 labels: ['TWF', 'HDF', 'PWF', 'OSF']


In [2]:
def predict_machine(machine_type, air_temp, process_temp, rpm, torque, tool_wear):
    # Basic input validation
    if machine_type not in ["L", "M"]:
        raise ValueError("machine_type must be 'L' or 'M'.")

    if air_temp <= 0:
        raise ValueError("air_temp must be positive.")

    if process_temp <= 0:
        raise ValueError("process_temp must be positive.")

    if rpm <= 0:
        raise ValueError("rpm must be positive.")

    if torque < 0:
        raise ValueError("torque cannot be negative.")

    if tool_wear < 0:
        raise ValueError("tool_wear cannot be negative.")

    row = pd.DataFrame([{
        "Type": machine_type,
        "Air temperature [K]": air_temp,
        "Process temperature [K]": process_temp,
        "Rotational speed [rpm]": rpm,
        "Torque [Nm]": torque,
        "Tool wear [min]": tool_wear
    }])

    # Model 1: machine failure prediction
    x1 = prep1.transform(row)
    failure_probability = float(model1.predict_proba(x1)[:, 1][0])
    failed = failure_probability >= meta1["threshold"]

    result = {
        "failure_probability": failure_probability,
        "failure": bool(failed),
        "failure_types": []
    }

    # Model 2: failure-type prediction
    if failed:
        x2 = prep2.transform(row)
        type_pred = model2.predict(x2)[0]

        detected_types = [
            label
            for label, flag in zip(meta2["labels"], type_pred)
            if flag == 1
        ]

        result["failure_types"] = detected_types

        if detected_types:
            result["failure_type_status"] = "Specific failure type(s) identified."
        else:
            result["failure_type_status"] = (
                "Failure detected, but no specific failure type was identified."
            )
    else:
        result["failure_type_status"] = "No failure predicted."

    # Project-level risk rule
    # These thresholds are project rules, not industry standards.
    p = failure_probability
    if p < 0.20:
        risk, action = "LOW", "Continue monitoring"
    elif p < 0.35:
        risk, action = "MEDIUM", "Inspect during planned maintenance"
    elif p < 0.60:
        risk, action = "HIGH", "Schedule inspection soon"
    else:
        risk, action = "CRITICAL", "Prioritize immediate inspection"

    result["risk"] = risk
    result["maintenance_action"] = action

    return result


### Input validation and failure-type handling

The inference function validates the basic input ranges before prediction. When Model 1 predicts a failure, Model 2 attempts to identify one or more failure types. If no specific type is predicted, the result explicitly reports that the failure type is not identified.

The risk thresholds are project-level rules, not industry standards.


In [3]:
# Example input
example = predict_machine(
    machine_type="L",
    air_temp=302.0,
    process_temp=311.0,
    rpm=1400,
    torque=55.0,
    tool_wear=180
)

print(example)


{'failure_probability': 0.05, 'failure': False, 'failure_types': [], 'failure_type_status': 'No failure predicted.', 'risk': 'LOW', 'maintenance_action': 'Continue monitoring'}
